# Machine Learning Trading Signal with Alpaca

This notebook builds a simple machine learning trading signal using Alpaca market data, evaluates it with a backtest, and compares it against buy and hold.

**Important:** This homework uses Alpaca **paper trading only**. No real money is used.

## 1. Imports and Setup

The user can choose a ticker such as AAPL, MSFT, SPY, QQQ, or NVDA.

In [ ]:
import os
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.data.enums import DataFeed

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

load_dotenv()

API_KEY = os.getenv("ALPACA_API_KEY")
SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")

if not API_KEY or not SECRET_KEY:
    raise ValueError("Missing Alpaca keys. Create a .env file with ALPACA_API_KEY and ALPACA_SECRET_KEY.")

print("This is paper trading only — no real money is used.")

In [ ]:
# Choose a ticker. If input does not work in your environment, replace this with ticker = "AAPL".
ticker = input("Choose a ticker, for example AAPL, MSFT, SPY, QQQ, NVDA: ").upper().strip()

if ticker == "":
    ticker = "AAPL"

print("Ticker selected:", ticker)

## 2. Data Collection from Alpaca

This section downloads 5 years of daily OHLCV data from Alpaca and stores it in a Pandas DataFrame.

In [ ]:
client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

end_date = datetime.now(timezone.utc)
start_date = end_date - timedelta(days=365 * 5)

request = StockBarsRequest(
    symbol_or_symbols=ticker,
    timeframe=TimeFrame.Day,
    start=start_date,
    end=end_date,
    feed=DataFeed.IEX,
)

bars = client.get_stock_bars(request).df

if bars.empty:
    raise ValueError("No data returned. Check ticker, Alpaca keys, and data permissions.")

if isinstance(bars.index, pd.MultiIndex):
    df = bars.loc[ticker].copy()
else:
    df = bars.copy()

df = df.sort_index()
df = df.rename(columns={
    "open": "Open",
    "high": "High",
    "low": "Low",
    "close": "Close",
    "volume": "Volume",
})

print(df.shape)
df.head()

## 3. Feature Engineering

The homework asks for at least 6 technical indicators across categories. This notebook computes more than 6:

- Trend: SMA, EMA, MACD, ADX
- Momentum: RSI, Stochastic, Williams %R
- Volatility: Bollinger Bands, ATR
- Volume: OBV, CMF
- Also: log returns, rolling mean, and rolling standard deviation

In [ ]:
def compute_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()

    # Log returns
    data["Log_Return"] = np.log(data["Close"] / data["Close"].shift(1))

    # Rolling mean and rolling standard deviation
    data["Rolling_Mean_10"] = data["Close"].rolling(10).mean()
    data["Rolling_Std_10"] = data["Close"].rolling(10).std()

    # Trend indicators: SMA and EMA
    data["SMA_20"] = data["Close"].rolling(20).mean()
    data["SMA_50"] = data["Close"].rolling(50).mean()
    data["EMA_20"] = data["Close"].ewm(span=20, adjust=False).mean()

    # MACD
    ema_12 = data["Close"].ewm(span=12, adjust=False).mean()
    ema_26 = data["Close"].ewm(span=26, adjust=False).mean()
    data["MACD"] = ema_12 - ema_26
    data["MACD_Signal"] = data["MACD"].ewm(span=9, adjust=False).mean()

    # RSI
    delta = data["Close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    rs = avg_gain / avg_loss
    data["RSI"] = 100 - (100 / (1 + rs))

    # Stochastic Oscillator and Williams %R
    low_14 = data["Low"].rolling(14).min()
    high_14 = data["High"].rolling(14).max()
    data["Stochastic"] = 100 * (data["Close"] - low_14) / (high_14 - low_14)
    data["Williams_R"] = -100 * (high_14 - data["Close"]) / (high_14 - low_14)

    # Bollinger Bands
    rolling_20 = data["Close"].rolling(20)
    data["BB_Middle"] = rolling_20.mean()
    data["BB_Upper"] = data["BB_Middle"] + 2 * rolling_20.std()
    data["BB_Lower"] = data["BB_Middle"] - 2 * rolling_20.std()

    # ATR
    high_low = data["High"] - data["Low"]
    high_close = np.abs(data["High"] - data["Close"].shift())
    low_close = np.abs(data["Low"] - data["Close"].shift())
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    data["ATR"] = true_range.rolling(14).mean()

    # OBV
    data["OBV"] = (np.sign(data["Close"].diff()) * data["Volume"]).fillna(0).cumsum()

    # CMF
    money_flow_multiplier = ((data["Close"] - data["Low"]) - (data["High"] - data["Close"])) / (data["High"] - data["Low"])
    money_flow_volume = money_flow_multiplier.replace([np.inf, -np.inf], np.nan).fillna(0) * data["Volume"]
    data["CMF"] = money_flow_volume.rolling(20).sum() / data["Volume"].rolling(20).sum()

    # ADX simplified calculation
    up_move = data["High"].diff()
    down_move = -data["Low"].diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0)
    plus_di = 100 * pd.Series(plus_dm, index=data.index).rolling(14).sum() / true_range.rolling(14).sum()
    minus_di = 100 * pd.Series(minus_dm, index=data.index).rolling(14).sum() / true_range.rolling(14).sum()
    dx = 100 * np.abs(plus_di - minus_di) / (plus_di + minus_di)
    data["ADX"] = dx.rolling(14).mean()

    return data


df_features = compute_features(df)
df_features = df_features.replace([np.inf, -np.inf], np.nan).dropna()
print(df_features.shape)
df_features.head()

## 4. Define Target Variable

The binary target is whether the next-day return is positive.

In [ ]:
df_features["Next_Return"] = df_features["Close"].pct_change().shift(-1)
df_features["Target"] = (df_features["Next_Return"] > 0).astype(int)
df_features = df_features.dropna()

df_features[["Close", "Next_Return", "Target"]].head()

## 5. PCA

The features are standardized first. Then PCA is fitted. The notebook keeps enough components to explain at least 80% of the variance.

In [ ]:
feature_cols = [
    "Log_Return", "Rolling_Mean_10", "Rolling_Std_10",
    "SMA_20", "SMA_50", "EMA_20",
    "MACD", "MACD_Signal", "RSI",
    "Stochastic", "Williams_R",
    "BB_Middle", "BB_Upper", "BB_Lower",
    "ATR", "OBV", "CMF", "ADX",
]

X = df_features[feature_cols]
y = df_features["Target"]

split_idx = int(len(df_features) * 0.70)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca_full = PCA()
pca_full.fit(X_train_scaled)

cum_var = np.cumsum(pca_full.explained_variance_ratio_)
n_components = np.argmax(cum_var >= 0.80) + 1

pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Number of PCA components kept:", n_components)
print("Cumulative explained variance:", cum_var[n_components - 1])

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cum_var) + 1), cum_var, marker="o")
plt.axhline(0.80, linestyle="--")
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Cumulative Explained Variance")
plt.grid(True)
plt.savefig("charts/pca_variance.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Machine Learning Model and Signal

This notebook uses a Random Forest classifier. The signal rule is:

- Long if predicted probability > 0.6
- Flat if predicted probability <= 0.6

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42,
    class_weight="balanced",
)

model.fit(X_train_pca, y_train)

test_probs = model.predict_proba(X_test_pca)[:, 1]
test_preds = (test_probs > 0.60).astype(int)

accuracy = accuracy_score(y_test, test_preds)
print("Test Accuracy:", accuracy)
print(classification_report(y_test, test_preds))

test_df = df_features.iloc[split_idx:].copy()
test_df["Probability_Up"] = test_probs
test_df["Signal"] = np.where(test_df["Probability_Up"] > 0.60, 1, 0)

test_df[["Close", "Probability_Up", "Signal"]].head()

## 7. Backtest

Backtest rules:

- Initial capital: $100,000
- Long-only
- No leverage
- No short selling
- Uses previous day's signal to avoid look-ahead bias

In [ ]:
initial_capital = 100000

test_df["Market_Return"] = test_df["Close"].pct_change()

# Use yesterday's signal to avoid look-ahead bias
test_df["Strategy_Return"] = test_df["Signal"].shift(1) * test_df["Market_Return"]

test_df = test_df.dropna()

test_df["Buy_Hold_Value"] = initial_capital * (1 + test_df["Market_Return"]).cumprod()
test_df["ML_Strategy_Value"] = initial_capital * (1 + test_df["Strategy_Return"]).cumprod()

test_df["Trade"] = test_df["Signal"].diff().abs().fillna(0)
test_df["P&L"] = test_df["ML_Strategy_Value"].diff().fillna(0)

print("Number of trades:", int(test_df["Trade"].sum()))
test_df[["Close", "Signal", "Trade", "ML_Strategy_Value", "Buy_Hold_Value", "P&L"]].head()

## 8. Performance Metrics

The notebook computes total return, CAGR, volatility, Sharpe ratio, Sortino ratio, max drawdown, and win rate.

In [ ]:
def performance_metrics(returns: pd.Series, equity_curve: pd.Series) -> dict:
    returns = returns.dropna()
    equity_curve = equity_curve.dropna()

    total_return = equity_curve.iloc[-1] / equity_curve.iloc[0] - 1
    years = len(returns) / 252
    cagr = (equity_curve.iloc[-1] / equity_curve.iloc[0]) ** (1 / years) - 1 if years > 0 else np.nan
    volatility = returns.std() * np.sqrt(252)
    sharpe = (returns.mean() / returns.std()) * np.sqrt(252) if returns.std() != 0 else np.nan

    downside_returns = returns[returns < 0]
    sortino = (returns.mean() / downside_returns.std()) * np.sqrt(252) if downside_returns.std() != 0 else np.nan

    drawdown = equity_curve / equity_curve.cummax() - 1
    max_drawdown = drawdown.min()
    win_rate = (returns > 0).mean()

    return {
        "Total Return": total_return,
        "CAGR": cagr,
        "Volatility": volatility,
        "Sharpe Ratio": sharpe,
        "Sortino Ratio": sortino,
        "Max Drawdown": max_drawdown,
        "Win Rate": win_rate,
    }


ml_metrics = performance_metrics(test_df["Strategy_Return"], test_df["ML_Strategy_Value"])
bh_metrics = performance_metrics(test_df["Market_Return"], test_df["Buy_Hold_Value"])

metrics_df = pd.DataFrame({
    "ML Signal": ml_metrics,
    "Buy & Hold": bh_metrics,
})

metrics_df

## 9. Visualizations

The charts are saved to the `charts/` folder for GitHub submission.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(test_df.index, test_df["ML_Strategy_Value"], label="ML Signal")
plt.plot(test_df.index, test_df["Buy_Hold_Value"], label="Buy & Hold")
plt.xlabel("Date")
plt.ylabel("Portfolio Value")
plt.title(f"Equity Curve: {ticker}")
plt.legend()
plt.grid(True)
plt.savefig("charts/equity_curve.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
test_df["ML_Drawdown"] = test_df["ML_Strategy_Value"] / test_df["ML_Strategy_Value"].cummax() - 1
test_df["BH_Drawdown"] = test_df["Buy_Hold_Value"] / test_df["Buy_Hold_Value"].cummax() - 1

plt.figure(figsize=(10, 6))
plt.plot(test_df.index, test_df["ML_Drawdown"], label="ML Signal Drawdown")
plt.plot(test_df.index, test_df["BH_Drawdown"], label="Buy & Hold Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.title(f"Drawdown: {ticker}")
plt.legend()
plt.grid(True)
plt.savefig("charts/drawdown.png", dpi=300, bbox_inches="tight")
plt.show()

## 10. Final Notes for Video

In the video, show:

1. The code running
2. The equity curve, drawdown, and PCA variance chart
3. The backtest results table
4. The Alpaca paper trading dashboard
5. A paper trade or paper order in the dashboard

Say clearly:

> This is paper trading only — no real money is used.